# 🧠 RAG (Retrieval-Augmented Generation) 핵심 개념 총정리

이 노트북은 LLM이 모르는 외부 데이터(PDF 문서 등)를 검색하여 답변하게 만드는 **RAG 파이프라인**의 전체 과정을 담고 있습니다. 나중에 다시 보실 때 아래의 5단계 흐름을 기억하세요!

---

### 📄 1. Document Loaders (문서 불러오기)
- 외부 파일(PDF, TXT 등)을 파이썬이 읽을 수 있는 텍스트 데이터로 가져오는 과정입니다.
- `UnstructuredLoader`를 사용하여 표, 이미지, 단락을 구조적으로 똑똑하게 분리해서 가져옵니다.

### ✂️ 2. Text Splitters (문서 쪼개기 - Chunking)
- LLM은 한 번에 읽을 수 있는 글자 수(Context Window)에 제한이 있기 때문에 긴 문서를 작게 쪼개야 합니다.
- **글자 수(Character) 기준:** `max_characters`로 무식하게 자르는 방법.
- **토큰(Token) 기준:** `Tiktoken` 등을 사용하여 LLM이 실제로 인식하는 토큰 기준으로 자르는 방법 (문맥 유지에 더 유리함).

### 🗄️ 3. Embeddings & Vector Stores (임베딩과 벡터 저장소)
- **임베딩(Embedding):** 쪼개진 텍스트(청크)들의 '의미'를 수학적인 다차원 좌표(벡터)로 변환합니다. (`OllamaEmbeddings` 사용)
- **벡터 저장소(Vector Store):** 좌표로 변환된 청크들을 저장하는 데이터베이스입니다. (`Chroma` 사용) 캐싱(`CacheBackedEmbeddings`)을 적용하면 한 번 변환한 문서를 다시 계산하지 않아 속도가 매우 빨라집니다.

### 🔍 4. Retrievers (문서 검색기)
- 사용자가 질문을 던지면, 질문 역시 임베딩(좌표 변환)한 뒤 벡터 저장소에서 **거리가 가장 가까운(의미가 가장 유사한) 청크들**을 뽑아옵니다.
- 키워드 매칭이 아닌 '문맥(뉘앙스)' 기반의 똑똑한 검색을 수행합니다.

### ⛓️ 5. LCEL Chains (파이프라인 결합 & 답변 생성)
검색된 문서들을 바탕으로 LLM에게 프롬프트를 던져 최종 답변을 생성하는 방식입니다.
- **Stuff 방식:** 검색된 4~5개의 문서를 한 번에 몽땅 프롬프트에 구겨 넣고(Stuff) 한 번에 답변을 요구하는 가장 기본적이고 빠른 방식.
- **Map-Reduce 방식:** 문서가 너무 많을 때 사용하는 조별 과제 방식. 
  - `Map`: 각 문서마다 LLM을 따로따로 호출하여 관련 내용만 짧게 요약시킴.
  - `Reduce`: 요약된 노트들을 최종적으로 취합하여 하나의 완벽한 답변으로 완성함.

> **💡 주의사항 (Pitfalls)**
> - `retriever`가 반환하는 결과는 순수 텍스트가 아니라 메타데이터가 섞인 파이썬 객체(`[Document]`)입니다.
> - 프롬프트에 던져줄 때는 반드시 불필요한 쓰레기 데이터(`orig_elements` 등)를 제거하고 **순수 텍스트(`page_content`)만 뽑아서 포맷팅**해주는 과정(`format_docs` 함수)이 필수적입니다!


In [16]:
# 6.1 Data Loaders and Splitters

from langchain_ollama import ChatOllama
# from langchain_classic.document_loaders import UnstructuredFileLoader
# from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_unstructured import UnstructuredLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

# splitter = CharacterTextSplitter(
#     separator="\n",
#     chunk_size=600,
#     chunk_overlap=100,
# )

# loader_f = UnstructuredFileLoader("./files/chapter_one.pdf")
loader = UnstructuredLoader(
    "./files/chapter_one.pdf",
    chunking_strategy="basic",  # 의미 단위로 작은 조각들을 뭉쳐주는 옵션
    max_characters=600,
    overlap=100
)

# split 2
# docs_f = loader_f.load_and_split(text_splitter=splitter)

# UnstructuredLoader는 위에서 이미 chunking_strategy를 썼으므로 스플리터가 필요 없습니다.
docs = loader.load()

# 문서 내의 불필요한 빈 줄(\n\n)을 한 줄(\n)로 깔끔하게 정리합니다.
for doc in docs:
    doc.page_content = doc.page_content.replace("\n\n", "\n")

# 처음 3개의 청크(문서)를 하나의 for loop에서 나란히 비교해봅니다.
# for i, (doc, doc_f) in enumerate(zip(docs[:3], docs_f[:3])):  
for i, doc in enumerate(docs[:3]):  
    print(f"==== 📄 청크 {i+1} ====")
    
    print("[UnstructuredLoader]")
    clean_meta = {k: v for k, v in doc.metadata.items() if k != 'orig_elements'}
    print(f"🏷️ 메타데이터: {clean_meta}") 
    print(f"📝 내용:\n{doc.page_content}\n")
    
    # print("[UnstructuredFileLoader]")
    # clean_meta_f = {k: v for k, v in doc_f.metadata.items() if k != 'orig_elements'}
    # print(f"🏷️ 메타데이터: {clean_meta_f}") 
    # print(f"📝 내용:\n{doc_f.page_content}\n")
    # print("=" * 70)

==== 📄 청크 1 ====
[UnstructuredLoader]
🏷️ 메타데이터: {'source': './files/chapter_one.pdf', 'file_directory': './files', 'filename': 'chapter_one.pdf', 'last_modified': '2026-03-30T14:15:54', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/pdf', 'category': 'CompositeElement', 'element_id': '8167322d8b970f117655dfea35c66ae8'}
📝 내용:
chapter_one md
Part 1, Chapter 1
Part One
1 It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into
his breast in an e ort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions,
though not quickly enough to prevent a swirl of gritty dust from entering along with him.
The hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for
indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide:

==== 📄 청크 2 ====
[UnstructuredLoader]
🏷️ 메타데이터: {'source': './files/chapter_one.pdf', 'fil

In [19]:
# 6.2 Tiktoken

from langchain_ollama import ChatOllama
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_unstructured import UnstructuredLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,                 # 토큰 수
    chunk_overlap=100,              
)

loader = UnstructuredLoader(
    "./files/chapter_one.pdf",
    chunking_strategy="basic",
    max_characters=3000,            # 문자 수
    overlap=0
)

docs = loader.load_and_split(text_splitter=splitter)

for i, doc in enumerate(docs[:3]):
    print(f"==== 📄 청크 {i+1} ====")
    print(f"📝 내용:\n{doc.page_content}\n")
    print("=" * 70)

==== 📄 청크 1 ====
📝 내용:
chapter_one md
Part 1, Chapter 1
Part One
1 It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into
his breast in an e ort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions,
though not quickly enough to prevent a swirl of gritty dust from entering along with him.
The hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for
indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide:
the face of a man of about forty- ve, with a heavy black moustache and ruggedly handsome features.
Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working,
and at present the electric current was cut o  during daylight hours. It was part of the economy drive in
preparation for Hate Week. The  at was seven  ights up, and Winston, who was thirty-nine and had a


In [23]:
# 6.4 Vector Store

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_unstructured import UnstructuredLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

loader = UnstructuredLoader(
    "./files/chapter_one.pdf",
    chunking_strategy="basic",
    max_characters=3000,            # 문자 수
    overlap=0
)

docs = loader.load_and_split(text_splitter=splitter)
docs = filter_complex_metadata(docs)

embedders = OllamaEmbeddings(model="nomic-embed-text")

# 셀을 여러 번 실행해도 중복되지 않도록, 기존 컬렉션을 먼저 지워줍니다 (초기화)
temp_db = Chroma(collection_name="splitter_test")
try:
    temp_db.delete_collection()
except:
    pass  # 삭제할 컬렉션이 없으면 무시

vectorstores = Chroma.from_documents(docs, embedders, collection_name="splitter_test")

# vector = embedders.embed_query("Hi")
# len(vector)

# vector = embedders.embed_documents(["hi", "how", "are", "you longer sentences because"])
# print(len(vector), len(vector[0]))

/tmp/ipykernel_22001/989270113.py:28: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  temp_db = Chroma(collection_name="splitter_test")
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


In [25]:
results = vectorstores.similarity_search("where does winston live")
# len(results)
for i, doc in enumerate(results):
    print(f"==== 📄 검색된 청크 {i + 1} (길이: {len(doc.page_content)}자) ====")
    print(f"🏷️ 메타데이터:\n{doc.metadata}\n")
    print(f"📝 본문 내용:\n{doc.page_content}\n")
    print("-" * 70) # 구분선

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


==== 📄 검색된 청크 1 (길이: 2394자) ====
🏷️ 메타데이터:
{'filetype': 'application/pdf', 'filename': 'chapter_one.pdf', 'page_number': 1, 'file_directory': './files', 'languages': ['eng'], 'orig_elements': 'eJzNW12P28YV/SuDfUkL7Kgzw/n0W5IGsR8aB4nToEgDY764YpciFZKyogT97713SK0l77aI+iDIBgxdaUhreeace+69sz/9fpfbvMnd9L5Jd6/InWYqscpwmqzOVEbjqbeZ0ehMxepUa+HD3T252+TJJz95uOb3u9j3Q2o6P+WxxK0/9Lvp/To3D+sJ3rGSrxx3zsKFy2f7Jk1r+Eg5uXLKOQ0fbfumm/AOP/1UmRUs5nrFfr4nSyTUHHGmV/JZWNZCfDcexilv8Gf5tvk1t99vfcx3/4YP6qbN71Mz5Dj1wwEXrP6C7413y4ed32R8O679dsrD+77Lq22q78q3Hqf3mz41dZPLcxJMaMoqWrF3XL7i6pWSuG7rH/L7brcJeYBVHP/bKf86fXLXf+4Y/NkkvGI6bMt/+jr7BBfBFZ9CYpNkUlWSKqUDlbrS1EeZqKxY0lUSyWR/BUhktWL3RMn5qc+RlitVMBBiJZ6FZe0tQ/KtHybC78mX890JPwXkXTO1+SU8mKpql2tBAzNAEasr6pTPlCddGylqH1x1NTwsP8XD8RkAq55HZeXNo/G2y38IBMW5ND5qynMQoFOC0+C1p0bBX5mt8uJ6IHBmT1Hg3M4PXkkgxAtxWX7LUHDyZiJ7PxJPwoAPh8S+TST5A2k68vl2aNp74rtEpnUmse3j40j2echknIbmseke4INmmHLuVuTHphunviPfb5ppfU/WzUjiGu7S7X77rc0Jbjj1p5B/44fBT82H/A6/zAvQe8hK3HMFKsgd8C9YGpyIVFnjXB0cq2S6HvRi0bw

In [26]:
# 6.4 Vector Store (최신 방법)
from langchain_unstructured import UnstructuredLoader
from langchain_ollama import OllamaEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_classic.storage import LocalFileStore

cache_dir = LocalFileStore("./.cache/")

# 1. 텍스트 스플리터를 지우고, 로더 내부에 청킹 옵션을 직접 지시합니다.
loader = UnstructuredLoader(
    "./files/chapter_one.pdf", 
    chunking_strategy="basic", # 마법의 자체 청킹 모드
    max_characters=600,         # 최대 600자까지 알아서 묶어줌
    overlap=100
)

# 2. 스플리터 없이 바로 로드해버립니다!
docs = loader.load()

# 👇==== 새롭게 추가하는 마법의 '텍스트 정제' 로직 ====👇
for doc in docs:
    # \n\n 과 \n 을 전부 일반 띄어쓰기(' ')로 바꿔서 LLM이 읽기 좋게 다듬습니다.
    doc.page_content = doc.page_content.replace("\n\n", " ").replace("\n", " ")
# 👆========================================👆

docs = filter_complex_metadata(docs)

# 3. 임베딩 및 저장 (찌꺼기를 무시하기 위해 방 이름을 또 새롭게 바꿔줍니다)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstores = Chroma.from_documents(docs, cached_embeddings, collection_name="loader_test")

# 4. 결과 확인
results = vectorstores.similarity_search("where does winston live")

for i, doc in enumerate(results):
    print(f"==== 📄 검색된 청크 {i + 1} (길이: {len(doc.page_content)}자) ====")
    print(f"🏷️ 메타데이터:\n{doc.metadata}\n")
    print(f"📝 본문 내용:\n{doc.page_content}\n")
    print("-" * 70)


/home/gildellmint/Workspace/FULLSTACK-GPT/.venv/lib/python3.13/site-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


==== 📄 검색된 청크 1 (길이: 547자) ====
🏷️ 메타데이터:
{'orig_elements': 'eJzNl0tv4zYQx78K4bOpJcWXmNteFlugL6ABiiK7MChxaBHRw5DoOO6i371D2cEmm6BIL4Z9EPTnjEYUfxzP8O7bCjroYUib6Fc3ZCWcViC9o55XQGVpGHUOL1oY72xQykq9WpNVD8l5lxw+823VjOPk4+ASzIvu3HHcp00LcdsmHKkkLyy3tsIHz7ZD9KlFE8YrrLJWo2k3xiHlCHd3UhRsTYwxBfu6Jk+yMoXKUgleyDf04o4Dq/k4J+jz9/weH6H7Y+caWP2DhhA72Pg4QZPG6Zgdig95bF6djYPrIQ83rdslmDbjAMXOh9Uy8zlt+tHHEGFZq5KVmjJBBbvl8oarGyWz385tYTPs+xom9OL5tQke8zqsUgtHcnBDAk/SWJC/xj1pXb4nXXwAQinx0S/3axKmsUdrHRNJrUukhgYnR+Iwpzg0KTvHAU1A3Dzv+12K43DyhAfAj8vvPe6Wr/nVTZNLGPU2TwRn9CN2xUWjgmLUWyWoVMbTSjFLK4CKMc+CCPxy2K04cT3JisnzLpC60K/1yf2asc/jfvAEF4D0zgNugJmMiKgFN/k1cQNe4LGBXcpAvZvuB5jn9Qkj6dE1kyJzM+2RfPwbfPEc7qdxxCm/RbX2TgXNOZUBGJWuKqkNjFPjpbCh9GUI1QWolggoc+OqEJnbky7lSQvG39SL/zVz/fA+DI2vhDU15hV3JZUW06zmmQozFoQA3WhzAQyKL/+SXJ+y5yxLdZbKvCEX52tG8P2J5yw+A+bZmyyY1yI0JtBaB0elcIY6WQPVSlteldx47y/AQpii+o7ipJ6WnjP9nMRZXiOJ8jmJZ1G/7Bn+ev8+JBY7C89AUG60pLKpDLUNF1Q1xpalqMqaXSI9TsVGvig96lxplGCFfSXlFdadF0j+zL0CNgX3ubS0cSa1a+5J2k/D0n8svUNCGFh

In [ ]:
# 6.6 RetrievalQA

from langchain_ollama.chat_models import ChatOllama
from langchain_unstructured import UnstructuredLoader
from langchain_ollama import OllamaEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.vectorstores import Chroma  # FAISS
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_classic.storage import LocalFileStore
from langchain_classic.chains import RetrievalQA

llm = ChatOllama(
    model="llama3.1",
    temperature=0.1,
)

cache_dir = LocalFileStore("./.cache/")

loader = UnstructuredLoader(
    "./files/chapter_one.pdf", 
    chunking_strategy="basic",
    max_characters=1000,
    overlap=100
)

docs = loader.load()

for doc in docs:
    doc.page_content = doc.page_content.replace("\n\n", " ").replace("\n", " ")

docs = filter_complex_metadata(docs)

embeddings = OllamaEmbeddings(model="nomic-embed-text")
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

try:
    Chroma(collection_name="loader_test", embedding_function=cached_embeddings).delete_collection()
except:
    pass  # 삭제할 게 없으면 그냥 넘어갑니다.

vectorstores = Chroma.from_documents(docs, cached_embeddings, collection_name="loader_test")

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # refine, map_reduce, map_rerank
    retriever=vectorstores.as_retriever(),
)

# chain.run("Where does Winston live?")
print(chain.invoke({"query": "Where does Winston live?"})["result"])
print(chain.invoke({"query": "Describe Victory Mansions"})["result"])

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


Winston lives in Victory Mansions, which is a large apartment building. His exact address is not specified, but it's implied to be a typical residential area for Party members and citizens of Oceania.


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


According to the text, Victory Mansions is a residential building where Winston Smith lives. Here are some details about it:

* The hallway smells of boiled cabbage and old rag mats.
* There is a coloured poster on one end of the hallway, depicting an enormous face (later revealed to be Big Brother's).
* The building has stairs, but no working lift (elevator) during daylight hours due to the economy drive in preparation for Hate Week.


In [ ]:
# 6.8 Stuff LCEL Chain

import logging
logging.getLogger("pdfminer").setLevel(logging.ERROR)
logging.getLogger("unstructured").setLevel(logging.ERROR)

from langchain_ollama.chat_models import ChatOllama
from langchain_unstructured import UnstructuredLoader
from langchain_ollama import OllamaEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores import Chroma  # FAISS
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_classic.storage import LocalFileStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


llm = ChatOllama(
    model="llama3.1",
    temperature=0.1,
)

cache_dir = LocalFileStore("./.cache/")

loader = UnstructuredLoader(
    "./files/chapter_one.pdf",
    chunking_strategy="basic",
    max_characters=1000,
    overlap=100,
)

docs = loader.load()

for doc in docs:
    doc.page_content = doc.page_content.replace("\n\n", " ").replace("\n", " ")

docs = filter_complex_metadata(docs)

embeddings = OllamaEmbeddings(model="nomic-embed-text")
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

try:
    Chroma(
        collection_name="loader_test", embedding_function=cached_embeddings
    ).delete_collection()
except:
    pass  # 삭제할 게 없으면 그냥 넘어갑니다.

vectorstores = Chroma.from_documents(
    docs, cached_embeddings, collection_name="loader_test"
)

retriever = vectorstores.as_retriever()

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            # "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}",
            "You are a helpful assistant. Answer questions using only the following context.:\n\n{context}",
        ),
        ("human", "{question}"),
    ]
)

# 검색된 문서들에서 page_content 텍스트만 뽑아서 하나로 묶어줌
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
)

chain.invoke("Describe Victory Mansions").content

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


"Victory Mansions is a large apartment building where Winston Smith lives. The hallway inside the building smells of boiled cabbage and old rag mats, indicating that it may not be the most luxurious or well-maintained place. The building has stairs, as the elevator is often out of order due to the electric current being cut off during daylight hours as part of an economy drive in preparation for Hate Week.\n\nThe building's interior suggests a sense of drabness and austerity, which is consistent with the overall atmosphere of the dystopian society described in the novel."

### 🗺️ Map-Reduce 파이프라인 (조별 과제 모드)

> **"관련 없는 텍스트의 잡음(Noise)을 싹 걷어내고, 순도 높은 알맹이만 조장에게 넘기는 강력한 필터링 기술"**

이 방식은 텍스트를 무작정 한 번에 다 밀어 넣는 Stuff 방식과 달리, 2단계의 정교한 필터링 및 취합 과정을 거칩니다.

#### ✂️ 1. Map (조원들의 팩트 추출 및 필터링 단계)
- `map_doc_chain`: 조원 1명의 뇌 역할을 합니다. 문서 1장만 읽고 **"질문과 관련된 내용이 있으면 내 맘대로 요약하지 말고 원본 그대로(verbatim) 오려오고, 없으면 버려라!"** 라며 엄격하게 정보만 추출(Extraction)합니다.
- `map_docs`: 작업 반장 함수입니다. 조원들이 쓸데없는 내용을 다 버리고 쏙쏙 오려온 '관련 있는 엑기스 텍스트'들만 스태플러(`\n\n`)로 묶어서 정보의 밀도를 꽉꽉 압축합니다.

#### 👨‍🏫 2. Reduce (조장의 최종 답변 생성 단계)
- `final_prompt`: 조원들이 잡음을 다 걸러내고 가져온 순도 높은 엑기스 노트를 보고, 최종 답변을 작성하라는 조장의 지시서입니다.
- `chain`: 압축된 요약본(`map_chain`)을 조장(`llm`)에게 넘겨주어 가장 완벽하고 군더더기 없는 최종 정답을 도출하는 완성된 파이프라인입니다.

In [41]:
# 6.9 Map Reduce LCEL Chain

import logging

logging.getLogger("pdfminer").setLevel(logging.ERROR)
logging.getLogger("unstructured").setLevel(logging.ERROR)

from langchain_ollama.chat_models import ChatOllama
from langchain_unstructured import UnstructuredLoader
from langchain_ollama import OllamaEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores import Chroma  # FAISS
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_classic.storage import LocalFileStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


llm = ChatOllama(
    model="llama3.1",
    temperature=0.1,
)

cache_dir = LocalFileStore("./.cache/")

loader = UnstructuredLoader(
    "./files/chapter_one.pdf",
    chunking_strategy="basic",
    max_characters=1000,
    overlap=100,
)

docs = loader.load()

for doc in docs:
    doc.page_content = doc.page_content.replace("\n\n", " ").replace("\n", " ")

docs = filter_complex_metadata(docs)

embeddings = OllamaEmbeddings(model="nomic-embed-text")
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

try:
    Chroma(
        collection_name="loader_test", embedding_function=cached_embeddings
    ).delete_collection()
except:
    pass  # 삭제할 게 없으면 그냥 넘어갑니다.

vectorstores = Chroma.from_documents(
    docs, cached_embeddings, collection_name="loader_test"
)

retriever = vectorstores.as_retriever()

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            # "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}",
            "You are a helpful assistant. Answer questions using only the following context.:\n\n{context}",
        ),
        ("human", "{question}"),
    ]
)


map_doc_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """ 
            Use the following portion of a long document to see if any of the text is relevant to answer the question. Return any relevant text verbatim.
            ------
            {context}
            """,
        ),
        ("human", "{question}"),
    ]
)

map_doc_chain = map_doc_prompt | llm


def map_docs(inputs):
    # print("inputs", inputs)
    documents = inputs["documents"]
    question = inputs["question"]
    # results = []
    # for document in documents:
    #     result = map_doc_chain.invoke({
    #         "context": document.page_content,
    #         "question": question
    #     }).content
    #     results.append(result)
    # print("results", results)
    # results = "\n\n".join(results)
    # return results

    return "\n\n".join(
        map_doc_chain.invoke(
            {"context": doc.page_content, "question": question}
        ).content
        for doc in documents
    )


map_chain = {
    "documents": retriever,
    "question": RunnablePassthrough(),
} | RunnableLambda(map_docs)

final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Given the following extracted parts of a long document and a question, create a final answer.
            ------
            {context}
            """,
        ),
        ("human", "{question}"),
    ]
)

chain = {"context": map_chain, "question": RunnablePassthrough()} | final_prompt | llm

chain.invoke("Describe Victory Mansions").content

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


'Based on George Orwell\'s novel "1984", Victory Mansions is described as a large apartment building where Winston Smith lives with his family. The building has a hallway with a strong smell of boiled cabbage and old rag mats, and features a large coloured poster on the wall depicting an enormous face of a man with a heavy black moustache and ruggedly handsome features. Additionally, the building has stairs but no working lift due to power cuts during daylight hours as part of an "economy drive" in preparation for Hate Week.'